In [2]:
import pandas as pd
import itertools
import json
import unicodedata
import os
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# --- CONFIGURATION ---
# CHANGE THIS to the exact name of your uploaded CSV file for imperative sentences
input_filename = "Imperative sentence.csv"

# Paths
input_path = f"/content/drive/MyDrive/dataset/Imperative sentence - Sheet1.csv"
# Output Path
output_path = "/content/drive/MyDrive/dataset/marma_imperative_dataset.jsonl"

# 2. Load the Dataset
print("Loading dataset...")
try:
    df = pd.read_csv(input_path)
except FileNotFoundError:
    # Fallback checking the local Colab directory
    print(f"Could not find file in Drive. Checking local storage for '{input_filename}'...")
    try:
        df = pd.read_csv(input_filename)
    except FileNotFoundError:
        print("❌ Error: File not found! Please check the file name and path.")
        raise

print(f"Original Row Count: {len(df)}")

# 3. Define Cleaning Functions

def normalize_text(text):
    """Normalizes unicode characters (fixes mixed Bengali/Latin encodings)."""
    if pd.isna(text): return text
    return unicodedata.normalize('NFKC', str(text)).strip()

def add_punctuation(text, lang_type):
    """
    Adds '.' for English-script based text and '।' for Bangla-script based text
    ONLY if punctuation is missing. (Same logic as Assertive Sentences)
    """
    if not text: return ""
    text = str(text).strip()

    # Define end characters
    eng_ends = ('.', '!', '?', '"', "'")
    ben_ends = ('।', '!', '?', '"', "'")

    if lang_type == 'english':
        if not text.endswith(eng_ends):
            return text + "."
    elif lang_type == 'bangla':
        if not text.endswith(ben_ends):
            # Check if it accidentally has an English dot
            if text.endswith('.'):
                return text[:-1] + "।"
            return text + "।"

    return text

# 4. Apply Cleaning
print("Cleaning data (Removing duplicates, normalizing text, adding punctuation)...")

# Map columns to their script type for punctuation logic
col_script_map = {
    'English': 'english',
    'Marma_English_Transliterate': 'english',
    'Bangla': 'bangla',
    'Marma_Bangla_Transliterate': 'bangla'
}

# Clean each cell
for col, script_type in col_script_map.items():
    if col in df.columns:
        # 1. Normalize Unicode
        df[col] = df[col].apply(normalize_text)
        # 2. Add Punctuation
        df[col] = df[col].apply(lambda x: add_punctuation(x, script_type))

# Remove rows with any missing values (NaN) or empty strings
df.replace("", float("nan"), inplace=True)
df.dropna(inplace=True)

# Remove exact duplicate rows
df.drop_duplicates(inplace=True)

print(f"Cleaned Row Count: {len(df)}")

# 5. Generate JSONL Dataset (Alpaca Format)
print("Generating instruction pairs...")

# Map CSV Columns to "Human Readable" names for the prompt
lang_names = {
    'English': 'English',
    'Bangla': 'Bangla',
    'Marma_English_Transliterate': 'Marma (English Script)',
    'Marma_Bangla_Transliterate': 'Marma (Bangla Script)'
}

alpaca_data = []

for idx, row in df.iterrows():
    # Get values for all available columns
    available_langs = [col for col in lang_names.keys() if col in df.columns]

    # Generate ALL permutations (Eng->Marma, Marma->Eng, Bangla->Marma, etc.)
    for source_col, target_col in itertools.permutations(available_langs, 2):
        source_lang = lang_names[source_col]
        target_lang = lang_names[target_col]
        source_text = row[source_col]
        target_text = row[target_col]

        # Create the instruction entry
        entry = {
            "instruction": f"Translate this {source_lang} sentence into {target_lang}.",
            "input": source_text,
            "output": target_text
        }
        alpaca_data.append(entry)

# 6. Save to Google Drive
print(f"Saving {len(alpaca_data)} instructions to {output_path}...")

# Ensure directory exists
os.makedirs(os.path.dirname(output_path), exist_ok=True)

with open(output_path, 'w', encoding='utf-8') as f:
    for entry in alpaca_data:
        json.dump(entry, f, ensure_ascii=False)
        f.write('\n')

print("\n✅ Success! Dataset saved.")
print(f"File location: {output_path}")
print("You can now proceed to the training notebook.")

Loading dataset...
Original Row Count: 1649
Cleaning data (Removing duplicates, normalizing text, adding punctuation)...
Cleaned Row Count: 1627
Generating instruction pairs...
Saving 19524 instructions to /content/drive/MyDrive/dataset/marma_imperative_dataset.jsonl...

✅ Success! Dataset saved.
File location: /content/drive/MyDrive/dataset/marma_imperative_dataset.jsonl
You can now proceed to the training notebook.
